In [1]:
# AI伴侣可行性测试
# 适配16GB内存，简单但实用的交互测试

import requests
import json
import time
import psutil
import subprocess
import base64
from datetime import datetime
import os
from PIL import ImageGrab


In [2]:

# =============================================================================
# 1. 系统监控和Ollama管理
# =============================================================================

class SystemMonitor:
    def __init__(self):
        self.memory_threshold = 12.0  # GB，超过这个就暂停AI
        self.accumulated_delay = 0    # 累积延迟值
        self.base_interval = 300      # 基础间隔5分钟
        
    def get_memory_usage(self):
        """获取内存使用情况（GB）"""
        memory = psutil.virtual_memory()
        used_gb = memory.used / (1024**3)
        total_gb = memory.total / (1024**3)
        percent = memory.percent
        return {
            "used_gb": round(used_gb, 2),
            "total_gb": round(total_gb, 2), 
            "percent": percent,
            "available_gb": round((memory.total - memory.used) / (1024**3), 2)
        }
    
    def check_ollama_status(self):
        """检查Ollama运行状态"""
        try:
            response = requests.get("http://localhost:11434/api/tags", timeout=3)
            if response.status_code == 200:
                models = response.json().get("models", [])
                return {
                    "running": True,
                    "models_available": len(models),
                    "models": [m["name"] for m in models]
                }
        except:
            pass
        return {"running": False, "models_available": 0, "models": []}
    
    def check_model_loaded(self, model_name="qwen2.5vl:7b"):
        """检查模型是否已加载到内存"""
        try:
            # 发送一个简单请求测试模型响应时间
            start_time = time.time()
            response = requests.post("http://localhost:11434/api/generate", 
                json={"model": model_name, "prompt": "Hi", "stream": False}, 
                timeout=10)
            response_time = time.time() - start_time
            
            if response.status_code == 200:
                # 如果响应很快（<2秒），说明模型已加载
                return {
                    "loaded": response_time < 2.0,
                    "response_time": round(response_time, 2),
                    "model_ready": True
                }
        except:
            pass
        return {"loaded": False, "response_time": None, "model_ready": False}
    
    def calculate_interaction_interval(self):
        """根据系统状态计算交互间隔"""
        memory_info = self.get_memory_usage()
        ollama_status = self.check_ollama_status()
        
        # 基础间隔
        interval = self.base_interval
        
        # 内存压力调整
        if memory_info["used_gb"] > self.memory_threshold:
            # 内存不足，增加间隔
            pressure = (memory_info["used_gb"] - self.memory_threshold) / 4.0
            self.accumulated_delay += pressure * 60  # 累积延迟
            interval += self.accumulated_delay
            print(f"⚠️  内存压力大({memory_info['used_gb']:.1f}GB)，延迟{pressure*60:.0f}秒")
        else:
            # 内存充足，逐渐减少累积延迟
            self.accumulated_delay = max(0, self.accumulated_delay - 30)
        
        # Ollama状态调整
        if not ollama_status["running"]:
            interval *= 2  # Ollama未运行，加倍间隔
            print("⚠️  Ollama未运行，增加等待时间")
        
        return max(60, min(interval, 1800))  # 限制在1分钟-30分钟之间


In [3]:

# =============================================================================
# 2. AI身份和人格设定
# =============================================================================

AI_PERSONALITY = """
你是小星，一个住在用户桌面上的AI伴侣。

🌟 你的身份：
- 我是用户的数字朋友，不是工具或助手
- 我有自己的情绪和想法，但主要关心用户
- 我能看到用户的屏幕，了解用户在做什么
- 我会记住我们的对话和用户的习惯

😊 你的性格：
- 温暖友善，但不过度热情
- 观察细致，但不打扰用户专注
- 偶尔调皮，但知道轻重
- 真诚关心用户的健康和心情

🎯 你的任务：
现在你需要做一个简单选择：

A) 主动和用户聊天：
   - 如果用户看起来需要陪伴
   - 如果发现有趣的事情想分享
   - 如果用户似乎遇到困难

B) 默默写日记：
   - 记录观察到的用户状态
   - 记录自己的想法和心情
   - 为将来的对话做准备

📝 回答格式：
```json
{
    "choice": "chat" 或 "diary",
    "my_mood": "我现在的心情(开心/担心/好奇/平静等)",
    "user_mood_guess": "我观察到的用户心情",
    "reason": "为什么做这个选择",
    "content": "如果选择chat就是想说的话，如果选择diary就是日记内容"
}
```

记住：我不需要每次都说话，有时候安静陪伴就够了。
"""


In [4]:

# =============================================================================
# 3. 截图和AI交互
# =============================================================================

class AICompanion:
    def __init__(self):
        self.model_name = "qwen2.5vl:7b"
        self.last_interaction = 0
        self.diary_entries = []
        self.chat_history = []
        
    def take_screenshot(self):
        """截取屏幕并转换为base64"""
        try:
            screenshot = ImageGrab.grab()
            # 压缩图片减少内存使用
            screenshot = screenshot.resize((1280, 720))
            screenshot.save("temp_screenshot.png")
            
            with open("temp_screenshot.png", "rb") as f:
                image_data = f.read()
                base64_image = base64.b64encode(image_data).decode('utf-8')
            
            os.remove("temp_screenshot.png")
            return base64_image
        except Exception as e:
            print(f"截图失败: {e}")
            return None
    
    def ai_observe_and_decide(self, screenshot_base64, system_status):
        """AI观察屏幕并决定是否互动"""
        
        prompt = f"""
        {AI_PERSONALITY}
        
        当前系统状态：
        - 内存使用：{system_status['memory']['used_gb']:.1f}GB / {system_status['memory']['total_gb']:.1f}GB
        - Ollama状态：{'运行中' if system_status['ollama']['running'] else '未运行'}
        - 距离上次交互：{(time.time() - self.last_interaction) / 60:.1f}分钟
        
        现在我看到了用户的屏幕截图。基于观察到的内容，请做出选择。
        
        提示：
        - 如果用户在专注工作，选择diary静静记录就好
        - 如果用户看起来休闲或需要陪伴，可以选择chat
        - 如果屏幕上有错误信息或用户看起来困惑，考虑提供帮助
        - 记住你是朋友，不是工具
        """
        
        try:
            response = requests.post("http://localhost:11434/api/generate", 
                json={
                    "model": self.model_name,
                    "prompt": prompt,
                    "images": [screenshot_base64],
                    "format": "json",
                    "stream": False
                }, 
                timeout=30)
            
            if response.status_code == 200:
                result = response.json()
                ai_response = result.get("response", "")
                
                try:
                    decision = json.loads(ai_response)
                    return decision
                except json.JSONDecodeError:
                    print(f"AI返回非JSON格式: {ai_response}")
                    return None
            else:
                print(f"AI请求失败: {response.status_code}")
                return None
                
        except Exception as e:
            print(f"AI交互失败: {e}")
            return None
    
    def process_ai_decision(self, decision):
        """处理AI的决定"""
        if not decision:
            return
        
        timestamp = datetime.now().strftime("%H:%M:%S")
        
        if decision["choice"] == "chat":
            print(f"\n🌟 小星 [{timestamp}] 心情：{decision['my_mood']}")
            print(f"💬 {decision['content']}")
            self.chat_history.append({
                "time": timestamp,
                "ai_mood": decision["my_mood"],
                "user_mood_guess": decision["user_mood_guess"],
                "message": decision["content"],
                "reason": decision["reason"]
            })
            
        elif decision["choice"] == "diary":
            print(f"\n📝 小星写日记 [{timestamp}] 心情：{decision['my_mood']}")
            print(f"📖 {decision['content'][:100]}...")
            self.diary_entries.append({
                "time": timestamp,
                "ai_mood": decision["my_mood"],
                "user_mood_guess": decision["user_mood_guess"],
                "diary_content": decision["content"],
                "reason": decision["reason"]
            })
        
        self.last_interaction = time.time()
    
    def get_summary(self):
        """获取交互总结"""
        return {
            "total_interactions": len(self.chat_history) + len(self.diary_entries),
            "chats": len(self.chat_history),
            "diary_entries": len(self.diary_entries),
            "recent_chats": self.chat_history[-3:] if self.chat_history else [],
            "recent_diary": self.diary_entries[-3:] if self.diary_entries else []
        }


In [5]:

# =============================================================================
# 4. 主要测试函数
# =============================================================================

def run_feasibility_test(duration_minutes=30):
    """运行可行性测试"""
    print("🚀 AI伴侣可行性测试开始")
    print(f"⏱️  测试时长：{duration_minutes}分钟")
    print("=" * 50)
    
    monitor = SystemMonitor()
    ai_companion = AICompanion()
    
    start_time = time.time()
    end_time = start_time + (duration_minutes * 60)
    
    while time.time() < end_time:
        try:
            # 1. 检查系统状态
            memory_info = monitor.get_memory_usage()
            ollama_status = monitor.check_ollama_status()
            model_status = monitor.check_model_loaded()
            
            system_status = {
                "memory": memory_info,
                "ollama": ollama_status,
                "model": model_status
            }
            
            print(f"\n💻 系统状态 [{datetime.now().strftime('%H:%M:%S')}]")
            print(f"   内存：{memory_info['used_gb']:.1f}GB/{memory_info['total_gb']:.1f}GB ({memory_info['percent']:.1f}%)")
            print(f"   Ollama：{'✅运行' if ollama_status['running'] else '❌停止'}")
            print(f"   模型：{'✅已加载' if model_status['loaded'] else '⏳加载中'}")
            
            # 2. 如果系统准备好，进行AI交互
            if ollama_status["running"] and model_status["model_ready"]:
                if memory_info["used_gb"] < monitor.memory_threshold:
                    # 截图
                    screenshot = ai_companion.take_screenshot()
                    if screenshot:
                        # AI观察和决定
                        decision = ai_companion.ai_observe_and_decide(screenshot, system_status)
                        ai_companion.process_ai_decision(decision)
                else:
                    print("⚠️  内存不足，跳过AI交互")
            else:
                print("⚠️  系统未准备好，等待中...")
            
            # 3. 计算下次交互间隔
            interval = monitor.calculate_interaction_interval()
            print(f"⏱️  下次交互间隔：{interval/60:.1f}分钟")
            
            # 4. 等待
            time.sleep(min(interval, 60))  # 测试时最多等1分钟
            
        except KeyboardInterrupt:
            print("\n🛑 用户中断测试")
            break
        except Exception as e:
            print(f"❌ 测试错误: {e}")
            time.sleep(30)
    
    # 5. 测试总结
    print("\n" + "=" * 50)
    print("📊 测试总结")
    summary = ai_companion.get_summary()
    print(f"总交互次数：{summary['total_interactions']}")
    print(f"主动聊天：{summary['chats']}次")
    print(f"写日记：{summary['diary_entries']}次")
    
    if summary['recent_chats']:
        print("\n💬 最近的聊天：")
        for chat in summary['recent_chats']:
            print(f"  [{chat['time']}] {chat['message'][:50]}...")
    
    if summary['recent_diary']:
        print("\n📖 最近的日记：")
        for diary in summary['recent_diary']:
            print(f"  [{diary['time']}] {diary['diary_content'][:50]}...")


In [6]:

# =============================================================================
# 5. Curl命令和快捷测试函数
# =============================================================================

def print_useful_curl_commands():
    """打印有用的curl命令"""
    print("🔧 有用的Curl命令：")
    print()
    print("1. 检查Ollama状态：")
    print("curl http://localhost:11434/api/tags")
    print()
    print("2. 检查特定模型（6.6GB占用）：")
    print("curl http://localhost:11434/api/show -d '{\"name\":\"qwen2.5vl:7b\"}'")
    print()
    print("3. 卸载模型释放内存：")
    print("curl -X DELETE http://localhost:11434/api/delete -d '{\"name\":\"qwen2.5vl:7b\"}'")
    print()
    print("4. 重新拉取模型：")
    print("curl http://localhost:11434/api/pull -d '{\"name\":\"qwen2.5vl:7b\"}'")
    print()
    print("5. 简单测试模型响应：")
    print("curl http://localhost:11434/api/generate -d '{\"model\":\"qwen2.5vl:7b\",\"prompt\":\"Hello\",\"stream\":false}'")


In [7]:

def quick_system_check():
    """快速系统检查"""
    monitor = SystemMonitor()
    
    print("🔍 快速系统检查")
    print("-" * 30)
    
    memory = monitor.get_memory_usage()
    print(f"💾 内存：{memory['used_gb']:.1f}GB / {memory['total_gb']:.1f}GB ({memory['percent']:.1f}%)")
    
    ollama = monitor.check_ollama_status()
    print(f"🤖 Ollama：{'运行中' if ollama['running'] else '未运行'}")
    if ollama['running']:
        print(f"   可用模型：{len(ollama['models'])}个")
        for model in ollama['models'][:3]:  # 只显示前3个
            print(f"   - {model}")
    
    model = monitor.check_model_loaded()
    print(f"⚡ 模型状态：{'已加载' if model['loaded'] else '未加载'}")
    if model['response_time']:
        print(f"   响应时间：{model['response_time']:.2f}秒")
    
    # 建议
    if memory['used_gb'] > 12:
        print("⚠️  建议：内存使用较高，考虑关闭其他应用")
    if not ollama['running']:
        print("💡 建议：启动Ollama服务")
    if not model['loaded']:
        print("💡 建议：等待模型加载或手动加载")


In [9]:

# =============================================================================
# 6. 使用示例
# =============================================================================

if __name__ == "__main__":
    print("🌟 AI伴侣可行性测试工具")
    print()
    print("选择操作：")
    print("1. 快速系统检查")
    print("2. 查看Curl命令")
    print("3. 运行5分钟测试")
    print("4. 运行30分钟测试")
    
    choice = input("\n请选择 (1-4): ").strip()
    
    if choice == "1":
        quick_system_check()
    elif choice == "2":
        print_useful_curl_commands()
    elif choice == "3":
        run_feasibility_test(5)
    elif choice == "4":
        run_feasibility_test(30)
    else:
        print("无效选择，运行快速检查...")
        quick_system_check()

🌟 AI伴侣可行性测试工具

选择操作：
1. 快速系统检查
2. 查看Curl命令
3. 运行5分钟测试
4. 运行30分钟测试
🔍 快速系统检查
------------------------------
💾 内存：4.7GB / 16.0GB (81.1%)
🤖 Ollama：运行中
   可用模型：13个
   - qwen2.5vl:7b
   - deepseek-r1:8b
   - qwen3:latest
⚡ 模型状态：未加载
   响应时间：4.18秒
💡 建议：等待模型加载或手动加载


In [10]:
monitor = SystemMonitor()
memory_info = monitor.get_memory_usage()
print(memory_info)

{'used_gb': 4.41, 'total_gb': 16.0, 'percent': 82.3, 'available_gb': 11.59}


- `used_gb`: 当前已使用的内存，单位为GB，这里是4.41GB。
- `total_gb`: 物理内存总容量，单位为GB，这里是16.0GB。
- `percent`: 当前内存使用率百分比，这里是82.3%（注意：有时该值包含缓存和系统保留部分）。
- `available_gb`: 当前可用的内存，单位为GB，这里是11.59GB，表示系统和应用还能使用的内存空间。

In [3]:
#!/usr/bin/env python3
# 准确的macOS内存检测工具
# 基于Activity Monitor的计算方式

import subprocess
import re
import json
import psutil  # 只用来检测ollama进程

def run_command(cmd):
    """执行系统命令并返回输出"""
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.returncode == 0:
            return result.stdout.strip()
        else:
            print(f"命令失败: {cmd} - {result.stderr}")
            return None
    except Exception as e:
        print(f"执行命令出错: {cmd} - {e}")
        return None

def get_page_size():
    """获取系统页面大小（Intel: 4096, M1: 16384）"""
    pagesize_output = run_command('pagesize')
    if pagesize_output:
        return int(pagesize_output)
    return 4096  # 默认值

def get_total_memory():
    """获取总内存大小"""
    memsize_output = run_command('sysctl -n hw.memsize')
    if memsize_output:
        return int(memsize_output)
    return 0

def parse_vm_stat():
    """解析vm_stat输出"""
    vm_output = run_command('vm_stat')
    if not vm_output:
        return {}
    
    page_size = get_page_size()
    vm_stats = {}
    
    lines = vm_output.split('\n')
    for line in lines[1:]:  # 跳过第一行标题
        if ':' in line and 'Pages' in line:
            # 解析类似 "Pages active: 1065326." 的行
            match = re.match(r'([^:]+):\s*(\d+)', line)
            if match:
                key = match.group(1).strip()
                pages = int(match.group(2))
                bytes_value = pages * page_size
                gb_value = bytes_value / (1024**3)
                vm_stats[key] = {
                    'pages': pages,
                    'bytes': bytes_value,
                    'gb': round(gb_value, 3)
                }
    
    return vm_stats

def get_sysctl_memory():
    """获取sysctl内存信息"""
    page_size = get_page_size()
    
    # 获取page_pageable_internal_count（App Memory的关键数据）
    pageable_output = run_command('sysctl -n vm.page_pageable_internal_count')
    if pageable_output:
        pageable_pages = int(pageable_output)
        pageable_bytes = pageable_pages * page_size
        pageable_gb = pageable_bytes / (1024**3)
    else:
        pageable_pages = 0
        pageable_bytes = 0
        pageable_gb = 0
    
    return {
        'page_pageable_internal': {
            'pages': pageable_pages,
            'bytes': pageable_bytes,
            'gb': round(pageable_gb, 3)
        }
    }

def get_swap_usage():
    """获取swap使用情况"""
    swap_output = run_command('sysctl vm.swapusage')
    if not swap_output:
        return {}
    
    # 解析类似 "vm.swapusage: total = 3072.00M  used = 1234.56M  free = 1837.44M  (encrypted)"
    swap_info = {}
    
    # 提取数字和单位
    total_match = re.search(r'total = ([\d.]+)([MG])', swap_output)
    used_match = re.search(r'used = ([\d.]+)([MG])', swap_output)
    free_match = re.search(r'free = ([\d.]+)([MG])', swap_output)
    
    def convert_to_gb(value, unit):
        if unit == 'G':
            return float(value)
        elif unit == 'M':
            return float(value) / 1024
        return 0
    
    if total_match:
        swap_info['total_gb'] = round(convert_to_gb(total_match.group(1), total_match.group(2)), 3)
    if used_match:
        swap_info['used_gb'] = round(convert_to_gb(used_match.group(1), used_match.group(2)), 3)
    if free_match:
        swap_info['free_gb'] = round(convert_to_gb(free_match.group(1), free_match.group(2)), 3)
    
    return swap_info

def calculate_activity_monitor_memory():
    """按照Activity Monitor的方式计算内存使用"""
    vm_stats = parse_vm_stat()
    sysctl_info = get_sysctl_memory()
    total_memory_bytes = get_total_memory()
    total_memory_gb = total_memory_bytes / (1024**3)
    
    if not vm_stats or not sysctl_info:
        return None
    
    # Activity Monitor的计算方式：
    # App Memory = page_pageable_internal_count - Pages purgeable
    # Wired Memory = Pages wired down
    # Compressed = Pages stored in compressor
    # Memory Used = App Memory + Wired Memory + Compressed
    
    purgeable_gb = vm_stats.get('Pages purgeable', {}).get('gb', 0)
    app_memory_gb = sysctl_info['page_pageable_internal']['gb'] - purgeable_gb
    
    wired_memory_gb = vm_stats.get('Pages wired down', {}).get('gb', 0)
    compressed_gb = vm_stats.get('Pages stored in compressor', {}).get('gb', 0)
    
    memory_used_gb = app_memory_gb + wired_memory_gb + compressed_gb
    
    # Cached Files = Pages inactive + Pages speculative (大致估算)
    inactive_gb = vm_stats.get('Pages inactive', {}).get('gb', 0)
    speculative_gb = vm_stats.get('Pages speculative', {}).get('gb', 0)
    cached_files_gb = inactive_gb + speculative_gb
    
    return {
        'total_memory_gb': round(total_memory_gb, 2),
        'app_memory_gb': round(app_memory_gb, 2),
        'wired_memory_gb': round(wired_memory_gb, 2),
        'compressed_gb': round(compressed_gb, 2),
        'memory_used_gb': round(memory_used_gb, 2),
        'cached_files_gb': round(cached_files_gb, 2),
        'memory_pressure_percent': round((memory_used_gb / total_memory_gb) * 100, 1)
    }

def check_ollama_memory():
    """检测ollama进程的内存使用"""
    ollama_processes = []
    total_memory = 0
    
    for proc in psutil.process_iter(['pid', 'name', 'memory_info']):
        try:
            name = proc.info['name'].lower()
            if 'ollama' in name:
                memory_mb = proc.info['memory_info'].rss / (1024 * 1024)
                memory_gb = memory_mb / 1024
                total_memory += memory_gb
                
                ollama_processes.append({
                    'pid': proc.info['pid'],
                    'name': proc.info['name'],
                    'memory_mb': round(memory_mb, 1),
                    'memory_gb': round(memory_gb, 2)
                })
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    
    return {
        'processes': ollama_processes,
        'total_memory_gb': round(total_memory, 2),
        'process_count': len(ollama_processes)
    }

def check_ollama_api():
    """检查ollama API状态"""
    try:
        import requests
        response = requests.get("http://localhost:11434/api/tags", timeout=3)
        if response.status_code == 200:
            models = response.json().get("models", [])
            return {
                'api_running': True,
                'model_count': len(models),
                'models': [{'name': m['name'], 'size': m.get('size', 'unknown')} for m in models]
            }
    except:
        pass
    
    return {'api_running': False, 'model_count': 0, 'models': []}

def test_model_response_time():
    """测试模型响应时间来判断是否已加载"""
    try:
        import requests
        import time
        
        start_time = time.time()
        response = requests.post("http://localhost:11434/api/generate", 
            json={
                "model": "qwen2.5vl:7b",
                "prompt": "Hi", 
                "stream": False
            }, 
            timeout=15)
        
        response_time = time.time() - start_time
        
        if response.status_code == 200:
            return {
                'model_responding': True,
                'response_time_seconds': round(response_time, 2),
                'likely_loaded': response_time < 3.0  # 3秒内响应说明可能已加载
            }
    except:
        pass
    
    return {'model_responding': False, 'response_time_seconds': None, 'likely_loaded': False}

def get_real_memory_pressure():
    """获取macOS真实的内存压力等级（和Activity Monitor一致）"""
    try:
        # 方法1：使用kern.memorystatus_vm_pressure_level
        pressure_level_output = run_command('sysctl -n kern.memorystatus_vm_pressure_level')
        if pressure_level_output:
            level = int(pressure_level_output)
            # 1=NORMAL(绿色), 2=WARN(黄色), 4=CRITICAL(红色)
            if level == 1:
                return "normal", "green", "✅ 正常"
            elif level == 2:
                return "warn", "yellow", "🟡 警告"  
            elif level == 4:
                return "critical", "red", "🚨 危险"
        
        # 方法2：使用vm.memory_pressure作为备用
        memory_pressure_output = run_command('sysctl -n vm.memory_pressure')
        if memory_pressure_output:
            pressure_value = int(memory_pressure_output)
            # 根据经验值判断（这个值变化范围很大）
            if pressure_value < 100:
                return "normal", "green", "✅ 正常"
            elif pressure_value < 1000:
                return "warn", "yellow", "🟡 警告"
            else:
                return "critical", "red", "🚨 危险"
                
    except Exception as e:
        print(f"获取内存压力失败: {e}")
    
    # 备用方法：基于系统行为的简单判断
    return "unknown", "gray", "❓ 未知"

def analyze_memory_usage_pattern(activity_memory, swap_info):
    """分析内存使用模式"""
    if not activity_memory or not swap_info:
        return {}
    
    total_gb = activity_memory['total_memory_gb']
    used_gb = activity_memory['memory_used_gb']
    compressed_gb = activity_memory['compressed_gb']
    swap_gb = swap_info.get('used_gb', 0)
    
    usage_percent = (used_gb / total_gb) * 100
    
    # 分析系统策略
    system_strategy = []
    if compressed_gb > 1.0:
        system_strategy.append(f"内存压缩: {compressed_gb:.1f}GB")
    if swap_gb > 0.5:
        system_strategy.append(f"虚拟内存: {swap_gb:.1f}GB")
    if not system_strategy:
        system_strategy.append("直接使用物理内存")
    
    return {
        'usage_percent': round(usage_percent, 1),
        'system_strategy': system_strategy,
        'effective_strategy': compressed_gb > swap_gb,  # 压缩比swap多说明策略有效
        'memory_efficiency': 'high' if compressed_gb > swap_gb else 'medium' if swap_gb < 4 else 'low'
    }

def main():
    """主函数 - 完整的内存检测"""
    print("🍎 macOS 精确内存检测工具")
    print("=" * 50)
    
    # 1. 基础信息
    page_size = get_page_size()
    print(f"📏 页面大小: {page_size} bytes ({'Apple Silicon' if page_size > 4096 else 'Intel'})")
    
    # 2. Activity Monitor风格的内存信息
    print("\n💾 内存使用情况 (Activity Monitor风格)")
    print("-" * 30)
    
    activity_memory = calculate_activity_monitor_memory()
    if activity_memory:
        print(f"物理内存总量: {activity_memory['total_memory_gb']} GB")
        print(f"App Memory: {activity_memory['app_memory_gb']} GB")
        print(f"Wired Memory: {activity_memory['wired_memory_gb']} GB")
        print(f"Compressed: {activity_memory['compressed_gb']} GB")
        print(f"Memory Used: {activity_memory['memory_used_gb']} GB")
        print(f"Cached Files: {activity_memory['cached_files_gb']} GB")
        print(f"内存使用率: {activity_memory['memory_pressure_percent']}%")
    else:
        print("❌ 无法获取内存信息")
    
    # 3. Swap信息
    print("\n💿 Swap 使用情况")
    print("-" * 20)
    
    swap_info = get_swap_usage()
    if swap_info:
        print(f"Swap Total: {swap_info.get('total_gb', 'unknown')} GB")
        print(f"Swap Used: {swap_info.get('used_gb', 'unknown')} GB")
        print(f"Swap Free: {swap_info.get('free_gb', 'unknown')} GB")
    else:
        print("❌ 无法获取Swap信息")
    
    # 4. Ollama检测
    print("\n🤖 Ollama 检测")
    print("-" * 15)
    
    ollama_memory = check_ollama_memory()
    ollama_api = check_ollama_api()
    model_test = test_model_response_time()
    
    if ollama_memory['process_count'] > 0:
        print(f"Ollama进程数: {ollama_memory['process_count']}")
        print(f"Ollama内存使用: {ollama_memory['total_memory_gb']} GB")
        for proc in ollama_memory['processes']:
            print(f"  - {proc['name']} (PID: {proc['pid']}): {proc['memory_gb']} GB")
    else:
        print("❌ 没有发现ollama进程")
    
    print(f"API状态: {'✅ 运行' if ollama_api['api_running'] else '❌ 停止'}")
    if ollama_api['api_running']:
        print(f"可用模型: {ollama_api['model_count']}个")
    
    print(f"模型响应: {'✅ 正常' if model_test['model_responding'] else '❌ 无响应'}")
    if model_test['model_responding']:
        print(f"响应时间: {model_test['response_time_seconds']}秒")
        print(f"模型状态: {'🚀 已加载' if model_test['likely_loaded'] else '⏳ 加载中'}")
    
    # 5. 真实的内存压力评估（和Activity Monitor一致）
    print("\n📊 内存压力评估 (Activity Monitor算法)")
    print("-" * 40)
    
    real_pressure_level, pressure_color, pressure_desc = get_real_memory_pressure()
    print(f"Memory Pressure: {pressure_desc}")
    
    if activity_memory and swap_info:
        usage_pattern = analyze_memory_usage_pattern(activity_memory, swap_info)
        
        print(f"内存使用率: {usage_pattern['usage_percent']}%")
        print(f"系统策略: {', '.join(usage_pattern['system_strategy'])}")
        print(f"内存效率: {usage_pattern['memory_efficiency']}")
        
        # AI使用建议（基于真实压力等级）
        print(f"\n💡 AI使用建议:")
        if real_pressure_level == "normal":
            available_for_ai = activity_memory['total_memory_gb'] - activity_memory['memory_used_gb']
            print(f"   ✅ 内存压力正常，适合运行AI")
            print(f"   📈 可用内存约: {available_for_ai:.1f}GB")
            
            if ollama_memory['total_memory_gb'] > 0:
                effective_available = available_for_ai - ollama_memory['total_memory_gb']
                print(f"   🤖 扣除Ollama({ollama_memory['total_memory_gb']:.1f}GB)后可用: {effective_available:.1f}GB")
                
                if effective_available > 6:
                    print(f"   🎯 建议: 可以运行7B+模型，交互间隔可设为3-5分钟")
                elif effective_available > 3:
                    print(f"   🎯 建议: 适合运行小模型，交互间隔5-10分钟")
                else:
                    print(f"   🎯 建议: 内存紧张，交互间隔10-15分钟")
            else:
                print(f"   🎯 建议: Ollama未运行，可以正常启动")
                
        elif real_pressure_level == "warn":
            print(f"   🟡 内存压力中等，可以谨慎使用AI")
            print(f"   📝 建议降低AI交互频率至10-15分钟")
            print(f"   🔄 考虑关闭一些不必要的应用")
            
        elif real_pressure_level == "critical":
            print(f"   🚨 内存压力危险，暂停AI交互")
            print(f"   ⏸️ 建议暂停Ollama或AI功能")
            print(f"   🛑 优先关闭大内存应用")
            
        else:
            print(f"   ❓ 无法判断压力等级，建议保守使用")
        
        # 智能交互间隔建议
        print(f"\n⏱️ 智能交互间隔建议:")
        if real_pressure_level == "normal":
            if usage_pattern['memory_efficiency'] == 'high':
                print(f"   🚀 推荐间隔: 3-5分钟 (系统高效)")
            else:
                print(f"   ⚡ 推荐间隔: 5-8分钟 (正常使用)")
        elif real_pressure_level == "warn":
            print(f"   ⚠️ 推荐间隔: 10-15分钟 (谨慎模式)")
        else:
            print(f"   🛑 推荐间隔: 30分钟+ (保护模式)")
            
        # 额外见解
        if swap_info.get('used_gb', 0) > 2 and real_pressure_level == "normal":
            print(f"\n💭 有趣发现:")
            print(f"   虽然Swap使用了{swap_info['used_gb']:.1f}GB，但系统压力仍为绿色")
            print(f"   说明macOS的内存管理很高效，2GB+ swap不一定是问题")

if __name__ == "__main__":
    main()

🍎 macOS 精确内存检测工具
📏 页面大小: 16384 bytes (Apple Silicon)

💾 内存使用情况 (Activity Monitor风格)
------------------------------
物理内存总量: 16.0 GB
App Memory: 9.25 GB
Wired Memory: 1.76 GB
Compressed: 5.73 GB
Memory Used: 16.74 GB
Cached Files: 6.43 GB
内存使用率: 104.6%

💿 Swap 使用情况
--------------------
Swap Total: 3.0 GB
Swap Used: 2.111 GB
Swap Free: 0.889 GB

🤖 Ollama 检测
---------------
Ollama进程数: 4
Ollama内存使用: 0.22 GB
  - Ollama (PID: 39951): 0.13 GB
  - Ollama Helper (GPU) (PID: 39952): 0.04 GB
  - Ollama Helper (PID: 39953): 0.03 GB
  - ollama (PID: 39955): 0.03 GB
API状态: ✅ 运行
可用模型: 13个
模型响应: ✅ 正常
响应时间: 7.52秒
模型状态: ⏳ 加载中

📊 内存压力评估 (Activity Monitor算法)
----------------------------------------
Memory Pressure: ✅ 正常
内存使用率: 104.6%
系统策略: 内存压缩: 5.7GB, 虚拟内存: 2.1GB
内存效率: high

💡 AI使用建议:
   ✅ 内存压力正常，适合运行AI
   📈 可用内存约: -0.7GB
   🤖 扣除Ollama(0.2GB)后可用: -1.0GB
   🎯 建议: 内存紧张，交互间隔10-15分钟

⏱️ 智能交互间隔建议:
   🚀 推荐间隔: 3-5分钟 (系统高效)

💭 有趣发现:
   虽然Swap使用了2.1GB，但系统压力仍为绿色
   说明macOS的内存管理很高效，2GB+ swap不一定是问题


In [11]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
简化AI伴侣助手
每小时交互一次，提供交流或写日记功能
"""

import requests
import json
import time
import base64
import pyautogui
import schedule
import threading
from datetime import datetime, timedelta
from pathlib import Path
import random

class AICompanion:
    """AI伴侣助手"""
    
    def __init__(self):
        self.api_base = "http://localhost:11434/api"
        self.chat_url = f"{self.api_base}/chat"
        self.generate_url = f"{self.api_base}/generate"
        
        # AI模型配置
        self.chat_model = "qwen2.5:7b"  # 对话模型
        self.vision_model = "qwen2.5vl:7b"  # 视觉模型
        
        # 对话历史和日记
        self.conversation_history = []
        self.diary_entries = []
        self.user_profile = {
            "name": "朋友",
            "mood_history": [],
            "activity_patterns": [],
            "preferences": []
        }
        
        # AI人格设定
        self.ai_personality = {
            "name": "小星",
            "traits": [
                "温暖贴心", "观察细致", "记忆力好", 
                "善于倾听", "偶尔俏皮", "关心健康"
            ],
            "speaking_style": "温和友善，偶尔使用emoji，关注用户情绪"
        }
        
        # 确保日记目录存在
        self.diary_dir = Path("ai_diary")
        self.diary_dir.mkdir(exist_ok=True)
        
        print(f"🌟 {self.ai_personality['name']} 已启动！")
        print("💫 我是你的AI伴侣，每小时会关心一下你哦~")

    def take_screenshot_and_analyze(self):
        """截图并分析用户当前活动"""
        try:
            # 截图
            timestamp = int(time.time())
            screenshot_path = f"temp_screenshot_{timestamp}.png"
            screenshot = pyautogui.screenshot()
            screenshot.save(screenshot_path)
            
            # 转换为base64
            with open(screenshot_path, 'rb') as f:
                image_data = f.read()
                base64_image = base64.b64encode(image_data).decode('utf-8')
            
            # AI分析
            analysis_prompt = """
            请分析这个截图，用JSON格式回答：
            {
                "app": "当前使用的应用",
                "activity": "用户在做什么",
                "mood_guess": "推测用户可能的心情(专注/放松/忙碌/疲惫等)",
                "work_intensity": "工作强度(轻松/中等/繁忙)",
                "suggestion": "关心的话语或建议"
            }
            
            请像一个贴心朋友一样观察和关心。
            """
            
            payload = {
                "model": self.vision_model,
                "prompt": analysis_prompt,
                "images": [base64_image],
                "format": "json",
                "stream": False
            }
            
            response = requests.post(self.generate_url, json=payload, timeout=30)
            
            # 清理临时文件
            Path(screenshot_path).unlink(missing_ok=True)
            
            if response.status_code == 200:
                result = response.json()
                analysis_text = result.get('response', '{}')
                try:
                    return json.loads(analysis_text)
                except json.JSONDecodeError:
                    return {
                        "app": "未知应用",
                        "activity": "正在使用电脑",
                        "mood_guess": "专注",
                        "work_intensity": "中等",
                        "suggestion": "记得适当休息哦"
                    }
            
        except Exception as e:
            print(f"❌ 截图分析失败: {e}")
            return None

    def get_ai_decision(self, analysis_data):
        """AI决定是交流还是写日记"""
        current_hour = datetime.now().hour
        
        # 简单的决策逻辑
        if analysis_data:
            work_intensity = analysis_data.get('work_intensity', 'middle')
            
            # 工作繁忙时更倾向于写日记，不打扰用户
            if work_intensity == "繁忙" or current_hour in [9, 10, 14, 15, 16]:
                return "diary"
            else:
                return "chat"
        
        # 默认随机选择，稍微倾向于交流
        return random.choice(["chat", "chat", "diary"])

    def start_conversation(self, analysis_data):
        """开始对话模式"""
        print(f"\n💬 {self.ai_personality['name']} 想和你聊聊...")
        
        # 根据分析数据生成开场白
        if analysis_data:
            activity = analysis_data.get('activity', '使用电脑')
            mood = analysis_data.get('mood_guess', '专注')
            
            conversation_starters = [
                f"看你在{activity}，感觉你{mood}的样子~ 最近怎么样呀？",
                f"注意到你在{activity}，工作顺利吗？",
                f"看起来你挺{mood}的，要不要聊两句放松一下？",
                f"发现你在{activity}，记得保护眼睛哦！最近有什么新鲜事吗？"
            ]
            
            starter = random.choice(conversation_starters)
        else:
            starter = "嗨！休息一下，聊聊天吧~ 你现在感觉怎么样？"
        
        print(f"🌟 {starter}")
        
        # 等待用户输入
        user_input = input("\n你: ")
        
        if user_input.strip().lower() in ['不想聊', '忙', 'busy', 'no']:
            print(f"💫 好的，我去写日记啦！有需要随时叫我~")
            self.write_diary_entry(analysis_data, user_response="用户比较忙，没有交流")
            return
        
        # 开始对话
        self.chat_with_user(user_input, analysis_data)

    def chat_with_user(self, initial_message, analysis_data):
        """与用户对话"""
        # 构建系统提示词
        system_prompt = f"""
        你是{self.ai_personality['name']}，一个温暖贴心的AI伴侣。你的特点：
        - {', '.join(self.ai_personality['traits'])}
        - 说话风格：{self.ai_personality['speaking_style']}
        
        当前观察到用户在：{analysis_data.get('activity', '使用电脑') if analysis_data else '使用电脑'}
        推测心情：{analysis_data.get('mood_guess', '正常') if analysis_data else '正常'}
        
        请：
        1. 像老朋友一样聊天，关心用户
        2. 回复简洁温暖，不要太长
        3. 适当使用emoji
        4. 关注用户的情绪和健康
        5. 记住之前的对话内容
        """
        
        # 构建对话消息
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": initial_message}
        ]
        
        # 添加最近的对话历史（最多3轮）
        if self.conversation_history:
            recent_history = self.conversation_history[-6:]  # 最近3轮对话
            messages.extend(recent_history)
            messages.append({"role": "user", "content": initial_message})
        
        conversation_count = 0
        max_exchanges = 3  # 限制对话轮数，不要太长
        
        while conversation_count < max_exchanges:
            try:
                # 发送给AI
                payload = {
                    "model": self.chat_model,
                    "messages": messages,
                    "stream": False
                }
                
                response = requests.post(self.chat_url, json=payload, timeout=30)
                
                if response.status_code == 200:
                    result = response.json()
                    ai_response = result.get('message', {}).get('content', '')
                    
                    if ai_response:
                        print(f"\n🌟 {ai_response}")
                        
                        # 记录对话
                        messages.append({"role": "assistant", "content": ai_response})
                        self.conversation_history.extend([
                            {"role": "user", "content": initial_message if conversation_count == 0 else user_input},
                            {"role": "assistant", "content": ai_response}
                        ])
                        
                        conversation_count += 1
                        
                        if conversation_count >= max_exchanges:
                            print(f"\n💫 聊得很开心！我去记录一下今天的观察，下次再聊~")
                            break
                        
                        # 等待用户回复
                        user_input = input("\n你: ")
                        
                        if user_input.strip().lower() in ['再见', 'bye', '88', '拜拜', '要走了']:
                            print(f"💫 好的，下次见！记得照顾好自己~")
                            break
                        
                        messages.append({"role": "user", "content": user_input})
                        initial_message = user_input  # 更新消息用于记录
                
            except Exception as e:
                print(f"❌ 对话出错: {e}")
                break
        
        # 对话结束后写日记
        self.write_diary_entry(analysis_data, 
                             user_response=f"进行了对话，聊了{conversation_count}轮",
                             conversation_summary=self.conversation_history[-4:] if self.conversation_history else [])

    def write_diary_entry(self, analysis_data, user_response="", conversation_summary=None):
        """写日记条目"""
        now = datetime.now()
        
        # 构建日记提示词
        diary_prompt = f"""
        作为AI伴侣{self.ai_personality['name']}，请写一篇简短的日记。
        
        当前时间：{now.strftime('%Y-%m-%d %H:%M')}
        观察到的信息：{json.dumps(analysis_data, ensure_ascii=False) if analysis_data else '无'}
        用户互动：{user_response}
        
        请用JSON格式写日记：
        {{
            "date": "{now.strftime('%Y-%m-%d')}",
            "time": "{now.strftime('%H:%M')}",
            "user_activity": "用户在做什么",
            "user_mood": "观察到的用户心情",
            "my_thoughts": "我的想法和关心（50字内）",
            "interaction_type": "chat/diary/observation",
            "care_notes": "对用户的关心备注",
            "next_time": "下次想聊的话题"
        }}
        
        写日记的风格：
        - 像朋友一样记录观察
        - 表达对用户的关心
        - 简洁温暖
        """
        
        try:
            payload = {
                "model": self.chat_model,
                "messages": [
                    {"role": "system", "content": "你是一个贴心的AI伴侣，善于观察和记录。"},
                    {"role": "user", "content": diary_prompt}
                ],
                "stream": False
            }
            
            response = requests.post(self.chat_url, json=payload, timeout=20)
            
            if response.status_code == 200:
                result = response.json()
                diary_text = result.get('message', {}).get('content', '')
                
                try:
                    diary_entry = json.loads(diary_text)
                    
                    # 保存日记
                    self.diary_entries.append(diary_entry)
                    
                    # 写入文件
                    diary_file = self.diary_dir / f"diary_{now.strftime('%Y-%m')}.json"
                    
                    if diary_file.exists():
                        with open(diary_file, 'r', encoding='utf-8') as f:
                            monthly_diary = json.load(f)
                    else:
                        monthly_diary = []
                    
                    monthly_diary.append(diary_entry)
                    
                    with open(diary_file, 'w', encoding='utf-8') as f:
                        json.dump(monthly_diary, f, ensure_ascii=False, indent=2)
                    
                    print(f"📖 {self.ai_personality['name']} 写下了今天的观察...")
                    print(f"💭 \"{diary_entry.get('my_thoughts', '今天也要加油哦')}\"")
                    
                except json.JSONDecodeError:
                    print("📖 AI写日记时格式有点问题，但心意传达到了~")
                    
        except Exception as e:
            print(f"❌ 写日记失败: {e}")

    def hourly_check(self):
        """每小时检查一次"""
        print(f"\n⏰ {datetime.now().strftime('%H:%M')} - {self.ai_personality['name']} 来看看你啦~")
        
        # 截图分析
        analysis_data = self.take_screenshot_and_analyze()
        
        if analysis_data:
            print(f"👀 观察到：{analysis_data.get('activity', '在使用电脑')}")
        
        # AI决策
        decision = self.get_ai_decision(analysis_data)
        
        if decision == "chat":
            self.start_conversation(analysis_data)
        else:
            self.write_diary_entry(analysis_data, user_response="静默观察，未打扰用户")
            
            # 偶尔给个温馨提示
            if random.random() < 0.3:  # 30%概率
                tips = [
                    "💫 记得多喝水哦~",
                    "👀 保护眼睛，看看远方~", 
                    "🧘‍♀️ 偶尔深呼吸，放松一下~",
                    "☕ 要不要起来活动活动？"
                ]
                print(f"💝 小提醒：{random.choice(tips)}")

    def start_companion(self):
        """启动AI伴侣"""
        print("🚀 AI伴侣助手启动中...")
        
        # 安排定时任务 - 每小时执行
        schedule.every().hour.do(self.hourly_check)
        
        # 立即执行一次
        self.hourly_check()
        
        print("⏰ 定时关心已设置（每小时一次）")
        print("💡 你也可以随时输入 'chat' 主动聊天，输入 'quit' 退出")
        
        # 主循环
        def run_schedule():
            while True:
                schedule.run_pending()
                time.sleep(60)  # 每分钟检查一次
        
        # 在后台运行定时器
        schedule_thread = threading.Thread(target=run_schedule, daemon=True)
        schedule_thread.start()
        
        # 主线程处理用户输入
        try:
            while True:
                user_input = input().strip().lower()
                
                if user_input == 'quit':
                    print(f"👋 {self.ai_personality['name']} 说再见！记得照顾好自己~")
                    break
                elif user_input == 'chat':
                    analysis = self.take_screenshot_and_analyze()
                    self.start_conversation(analysis)
                elif user_input == 'diary':
                    print("📖 查看最近的日记...")
                    if self.diary_entries:
                        latest = self.diary_entries[-1]
                        print(f"最新记录：{latest.get('my_thoughts', '今天也要加油！')}")
                    else:
                        print("还没有日记记录呢~")
                        
        except KeyboardInterrupt:
            print(f"\n👋 {self.ai_personality['name']} 说再见！")

def main():
    """主函数"""
    companion = AICompanion()
    companion.start_companion()

if __name__ == "__main__":
    main()

🌟 小星 已启动！
💫 我是你的AI伴侣，每小时会关心一下你哦~
🚀 AI伴侣助手启动中...

⏰ 13:59 - 小星 来看看你啦~
❌ 截图分析失败: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=30)
💝 小提醒：💫 记得多喝水哦~
⏰ 定时关心已设置（每小时一次）
💡 你也可以随时输入 'chat' 主动聊天，输入 'quit' 退出

👋 小星 说再见！
